In [1]:
# Application Description:
# FPGA-based hardware accelerator for ResMLP model.
# *** Software-only implementation for testing purposes ***

# Instructions:
# 1. Ensure you have the libraries installed (pip install timm torch pillow)
# 2. Run the jupyter notebook cells in order
# 3. Feel free to change the model name to any timm ResMLP model
# 4. Feel free to change the input image url to any image of your choice

# Performance:
# For the current single-image test, the bulk of the inference time is spent looping through the model layers
# ResMLP has no attention mechanism - all mixing is done via MLPs over tokens and channels

In [2]:
"""
Load the pre-trained ResMLP model
"""

from urllib.request import urlopen
from PIL import Image
import timm
import torch
import torch.nn.functional as F

model_name = 'resmlp_12_224.fb_in1k'

model = timm.create_model(model_name, pretrained=True)
model = model.eval()

stem = model.stem
blocks = model.blocks    # List of ResMLP layers
norm = model.norm        # Final affine norm layer
head = model.head        # Classification head (Linear)

# Get model-specific transforms (resize to 224x224, normalization, etc.)
data_config = timm.data.resolve_model_data_config(model)
transforms = timm.data.create_transform(**data_config, is_training=False)

print(f"Loaded model: {model_name}")
print(f"Num blocks: {len(blocks)}")
print(f"Hidden dim (num_features): {model.num_features}")

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded model: resmlp_12_224.fb_in1k
Num blocks: 12
Hidden dim (num_features): 384


In [3]:
"""
Software Step: Process image(s) for input
"""

# Load and convert image
url = 'https://hips.hearstapps.com/hmg-prod/images/schnauzer-carrying-lead-6643258113665.jpeg'
image = Image.open(urlopen(url)).convert("RGB")
input_tensor = transforms(image).unsqueeze(0)  # [1, 3, 224, 224]
print(f"Input tensor shape: {input_tensor.shape}")

# Patch embedding for input tokens
with torch.no_grad():
    x = stem(input_tensor)
    print(f"Shape after patch embedding: {x.shape}")
    # Output: [B, num_patches, hidden_dim] = [1, 196, 384]

Input tensor shape: torch.Size([1, 3, 224, 224])
Shape after patch embedding: torch.Size([1, 196, 384])


In [4]:
"""
Hardware Step: ResMLP Blocks
Each block has two sub-layers:
    (a) Token-mixing (Single Linear Layer) with Affine pre-norm and LayerScale
    (b) Channel-mixing (MLP) with Affine pre-norm and LayerScale
"""
with torch.no_grad():
    for i, block in enumerate(blocks):

        # (a) Token-Mixing Sublayer

        # 1. Affine pre-norm (Hardware: element-wise multiply + add, no statistics)
        norm1_alpha = block.norm1.alpha.detach()
        norm1_beta  = block.norm1.beta.detach()
        x_scaled_1  = (x * norm1_alpha) + norm1_beta

        # Transpose so Linear operates across the token (spatial) dimension
        x_scaled_1_T = x_scaled_1.transpose(1, 2)  # [B, hidden_dim, num_patches]

        # 2. Token-mixing Linear weights & biases (FPGA static weights)
        tok_w = block.linear_tokens.weight.detach()  # [196, 196]
        tok_b = block.linear_tokens.bias.detach()

        # Single linear projection (no activation in ResMLP token mixing)
        tok_out_T = torch.matmul(x_scaled_1_T, tok_w.T) + tok_b

        # 3. LayerScale 1: bare learned tensor multiplied onto output before residual
        ls1 = block.ls1.detach()
        tok_out = tok_out_T.transpose(1, 2)          # [B, num_patches, hidden_dim]
        x = x + ls1 * tok_out

        # (b) Channel-Mixing MLP Sublayer

        # 4. Affine pre-norm (Hardware: element-wise multiply + add, no statistics)
        norm2_alpha = block.norm2.alpha.detach()
        norm2_beta  = block.norm2.beta.detach()
        x_scaled_2  = (x * norm2_alpha) + norm2_beta

        # 5. Channel-mixing MLP weights & biases (FPGA static weights)
        ch_fc1_w = block.mlp_channels.fc1.weight.detach()
        ch_fc1_b = block.mlp_channels.fc1.bias.detach()
        ch_fc2_w = block.mlp_channels.fc2.weight.detach()
        ch_fc2_b = block.mlp_channels.fc2.bias.detach()

        # Two-layer MLP with GELU activation
        ch_hidden = F.gelu(torch.matmul(x_scaled_2, ch_fc1_w.T) + ch_fc1_b)
        ch_out    = torch.matmul(ch_hidden, ch_fc2_w.T) + ch_fc2_b

        # 6. LayerScale 2: bare learned tensor multiplied onto output before residual
        ls2 = block.ls2.detach()
        x = x + ls2 * ch_out

In [5]:
"""
Software Step: Final feature embedding
"""

with torch.no_grad():
    
    # Final layer norm
    # Equivalent to model.forward_features(transforms(image).unsqueeze(0))
    x = norm(x) # [B, num_patches, hidden_dim]

    # Global average pool over 196 tokens
    # Equivalent to model.forward_head(x, pre_logits=True)
    feature_embedding = x.mean(dim=1) # [B, hidden_dim]
    print(f"Shape of feature embedding: {feature_embedding.shape}")

Shape of feature embedding: torch.Size([1, 384])


In [6]:
"""
Optional: Classification
"""

ENABLE_CLASSIFICATION = True

if ENABLE_CLASSIFICATION:
    with torch.no_grad():
        # [1, 1000]
        logits = head(feature_embedding)
        pred_idx = logits.argmax(-1).item()

    from timm.data.imagenet_info import ImageNetInfo
    # 1000 classes
    imagenet_info = ImageNetInfo()
    label_descriptions = imagenet_info.label_descriptions(detailed=True, as_dict=False)

    pred_label = label_descriptions[pred_idx]
    print(f"Predicted class - {pred_label} (ID: {pred_idx})")

Predicted class - miniature schnauzer: a small schnauzer (ID: 196)
